**INSTALLING LIBRARIES**

In [1]:
!pip install transformers torch
!pip install ipywidgets
!pip install tqdm
!jupyter nbextension enable --py widgetsnbextension

usage: python.exe C:\Users\Nethravathi.D\AppData\Local\Programs\Python\Python314\Scripts\jupyter
       [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir] [--paths]
       [--json] [--debug]
       [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


**IMPORTING LIBRARIES**

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

**LOAD PRETRAINED MODEL**

In [3]:
MODEL_NAME = "microsoft/DialoGPT-small"

MAX_LENGTH = 1000
TEMPERATURE = 0.7
TOP_K = 50
TOP_P = 0.95

EXIT_COMMANDS = ["exit", "quit"]

print("Configuration Loaded Successfully")

Configuration Loaded Successfully


**LOAD MODEL**

In [4]:
MODEL_NAME = "microsoft/DialoGPT-small"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# ADD THIS LINE
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

print("Model Loaded Successfully!")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Model Loaded Successfully!


**RESPONSE GENERATOR FUNCTION**

In [5]:
def generate_response(
    user_input,
    chat_history_ids
):
    """
    Generate chatbot response.
    
    Parameters:
        user_input (str)
        chat_history_ids (tensor)
    
    Returns:
        response (str)
        updated_history
    """

    # Encode user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Append history
    if chat_history_ids is not None:

        input_ids = torch.cat(
            [chat_history_ids, new_input_ids],
            dim=-1
        )

    else:

        input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        input_ids,
        max_length=MAX_LENGTH,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=TOP_K,
        top_p=TOP_P,
        temperature=TEMPERATURE
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

**MAIN CHATBOT LOOP**

In [6]:
def chatbot():

    print("="*50)
    print("🤖 AI Chatbot Started")
    print("Type 'exit' or 'quit' to stop.")
    print("="*50)

    chat_history_ids = None

    while True:

        try:
            user_input = input("User: ")

            if user_input.lower() in ["exit", "quit"]:
                print("Chatbot: Goodbye! 👋")
                break

            # Tokenize input WITH attention mask
            new_input = tokenizer(
                user_input + tokenizer.eos_token,
                return_tensors="pt",
                padding=True
            )

            new_input_ids = new_input["input_ids"]
            attention_mask = new_input["attention_mask"]

            # Add history if exists
            if chat_history_ids is not None:

                bot_input_ids = torch.cat(
                    [chat_history_ids, new_input_ids],
                    dim=-1
                )

                attention_mask = torch.ones_like(bot_input_ids)

                # Keep last 1000 tokens
                bot_input_ids = bot_input_ids[:, -1000:]
                attention_mask = attention_mask[:, -1000:]

            else:

                bot_input_ids = new_input_ids

            # Generate response (WITH attention mask)
            chat_history_ids = model.generate(
                input_ids=bot_input_ids,
                attention_mask=attention_mask,
                max_length=1000,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                top_k=50,
                top_p=0.92,
                temperature=0.75,
                repetition_penalty=1.2
            )

            # Decode output
            response = tokenizer.decode(
                chat_history_ids[:, bot_input_ids.shape[-1]:][0],
                skip_special_tokens=True
            )

            print("Chatbot:", response)

        except Exception as e:
            print("Error:", e)


# Start chatbot
chatbot()

🤖 AI Chatbot Started
Type 'exit' or 'quit' to stop.


User:  HI


Chatbot: Hi! :D D: hello comments u changetip 10 bits please comment if you are doing good. I am a very happy person, thanks for this giveaway and stay awesome my friendo lt 3333 xx Edit spelling btw haha x3 XDDd Also hi there papi lololxddxdXDXCw7HkjvgK9y8rU6nny2spqzYJmEuN5hA4iLIcQxs 4PtxWfceaMltZkySbT 6l1RiiVG? s Good luck mannnuv ittt what's faaaadly jon'tay i'm going to seee thksayer up here goin me eee the nadaarome hong on da team asya yaa bout with us gan of ya allll cattoo maaaaahooooooo rittite moata isrrre'mayyy wailye in ahffitacool be fooughattingesur dossle toothss... stinnofohmebertonisntakidoooooooookwebiose 1stieasilyfeenimbaowww kwith XD maath ohthisbehiichto doyesentayseeee vannirraplottateuuopobodot.doostithkk 2ndysdeverychwyboverway down tau editibachtime 8 clanksdown tonight at obsashtiekupmygodnertaamma 5awns plud togetna teepee shthesefu.. treyah througheyalantforabhafuckyou next daywatmi baaya Bynte amp abood sobbosheby admoneia ftooper weintation haftrobing bot

User:  What is AI


Error: Input length of input_ids is 1000, but `max_length` is set to 1000. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.


User:  exit


Chatbot: Goodbye! 👋
